# Notebook 07: JAX and TPU Mental Model

## Why This Matters

JAX was designed at Google with TPUs as a primary target. Understanding how TPUs work -- their systolic array architecture, memory hierarchy, and XLA compilation model -- helps you write JAX code that is maximally efficient on TPUs. Many surprising behaviors (why bfloat16 is preferred, why padding matters, why dynamic shapes are problematic) become obvious once you understand the hardware.


In [ ]:
%pip install -q jax jaxlib numpy matplotlib

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import time
print("Backend:", jax.default_backend())
print("Devices:", jax.devices())
print("Ready.")

## 1. XLA: The Compiler Beneath JAX

**XLA (Accelerated Linear Algebra)** is Google's open-source ML compiler. JAX compiles to XLA, which then generates machine code for the target hardware.

XLA's optimization pipeline:
1. **Op fusion**: combine multiple elementwise ops into a single kernel (avoid memory round-trips)
2. **Layout optimization**: choose optimal memory layouts (row-major vs column-major) for each operation
3. **Memory planning**: reuse memory buffers where possible
4. **Loop tiling**: tile matrix multiply to fit in cache/SRAM

The key insight: **XLA needs to know shapes at compile time** to perform these optimizations. This is why dynamic shapes are problematic -- XLA cannot tile or fuse operations without knowing their sizes.


In [ ]:
# Demonstrate XLA fusion
# Without fusion: three separate kernels, three memory round-trips
# With fusion: XLA merges into one kernel

@jax.jit
def fused_ops(x):
    # XLA will fuse these into one kernel (no separate memory ops for sqrt, exp)
    return jnp.sin(jnp.sqrt(jnp.abs(x)))

def unfused_ops(x):
    # Without jit, each op dispatches separately
    a = jnp.abs(x)
    b = jnp.sqrt(a)
    return jnp.sin(b)

key = jax.random.PRNGKey(0)
x = jax.random.normal(key, (10_000, 1000))

_ = fused_ops(x).block_until_ready()  # warmup

t0 = time.perf_counter()
for _ in range(100):
    _ = fused_ops(x).block_until_ready()
t_fused = (time.perf_counter() - t0) / 100 * 1000

t0 = time.perf_counter()
for _ in range(100):
    _ = unfused_ops(x).block_until_ready()
t_unfused = (time.perf_counter() - t0) / 100 * 1000

print(f'JIT (fused):   {t_fused:.3f} ms')
print(f'Eager (unfused): {t_unfused:.3f} ms')
print(f'Fusion speedup: {t_unfused/t_fused:.2f}x')

## 2. TPU Architecture: Systolic Arrays

A TPU's core compute unit is a **systolic array** -- a 2D grid of multiply-accumulate (MAC) units that pass data in a wave pattern. This architecture is specifically optimized for matrix multiplication.

Key TPU properties:
- **Matrix multiply unit (MXU)**: 128x128 systolic array (v3/v4), performs 128x128 @ 128x128 matmul per cycle
- **High bandwidth memory (HBM)**: 1-3 TB/s bandwidth (vs GPU ~2 TB/s)
- **Vector unit**: handles elementwise ops, activations
- **Scalar unit**: control flow, loop management
- **CMEM (on-chip memory)**: tiny (8MB) but very fast -- model weights should be reused from here

**Implication**: TPUs are extremely fast at large matmuls but poor at irregular/sparse computation. A 128x128 matmul is the "sweet spot" -- smaller matmuls underutilize the MXU.


In [ ]:
# Why TPUs prefer large, padded matmuls
# Even on CPU/GPU, we can demonstrate the principle:
# Small matmuls are less efficient per FLOP than large ones

@jax.jit
def matmul(A, B):
    return jnp.dot(A, B)

sizes = [64, 128, 256, 512, 1024, 2048]
gflops_per_size = []

for n in sizes:
    A = jax.random.normal(jax.random.PRNGKey(0), (n, n))
    B = jax.random.normal(jax.random.PRNGKey(1), (n, n))
    
    _ = matmul(A, B).block_until_ready()  # warmup
    
    t0 = time.perf_counter()
    reps = 20
    for _ in range(reps):
        _ = matmul(A, B).block_until_ready()
    elapsed = (time.perf_counter() - t0) / reps
    
    flops = 2 * n**3  # matmul is 2*N^3 FLOPs
    gflops = flops / elapsed / 1e9
    gflops_per_size.append(gflops)
    print(f'N={n:5d}: {elapsed*1000:.3f} ms, {gflops:.1f} GFLOP/s')

plt.figure(figsize=(8, 4))
plt.plot(sizes, gflops_per_size, 'o-', color='steelblue')
plt.xlabel('Matrix size N')
plt.ylabel('GFLOP/s')
plt.title('Compute throughput vs matrix size (TPU prefers large N)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('matmul_throughput.png', dpi=100)
plt.show()

## 3. Why TPUs Want Static Shapes

XLA compiles **per (function, input shapes)**. When a new shape is encountered:
1. Python tracing runs again
2. XLA optimization pipeline runs again (often 10-60 seconds)
3. New compiled binary is cached

For TPU training with variable-length sequences, this means:
- Every unique sequence length triggers recompilation
- With 1000 unique lengths, you get 1000 compilations = hours wasted

**Solution**: pad all sequences to fixed maximum lengths. The cost: some wasted compute on padding tokens. The benefit: compile once, run fast forever.

$$\text{padding overhead} = \frac{\text{max\_len} - \text{mean\_len}}{\text{max\_len}}$$

Typical padding overhead for language models: 10-30% wasted compute, vastly outweighed by eliminating recompilations.


In [ ]:
# Static shapes: the correct TPU pattern
MAX_SEQ_LEN = 512
VOCAB_SIZE = 32000

def pad_to_length(ids, max_len, pad_id=0):
    length = len(ids)
    padded = ids + [pad_id] * (max_len - length)
    mask = [1] * length + [0] * (max_len - length)
    return jnp.array(padded), jnp.array(mask)

# Simulated sequence batch
sequences = [[1, 2, 3, 4], [5, 6], [7, 8, 9, 10, 11, 12, 13]]

print('Variable-length (BAD for TPU -- triggers recompilation per shape):')
for seq in sequences:
    print(f'  Shape: {jnp.array(seq).shape}')

print('\nPadded to fixed length (GOOD -- always same shape):')
for seq in sequences:
    padded, mask = pad_to_length(seq, 16)
    print(f'  padded: {padded}, mask: {mask}')

# bfloat16: native on TPU
x_f32 = jnp.array([1.0, 2.0, 3.0], dtype=jnp.float32)
x_bf16 = x_f32.astype(jnp.bfloat16)

print('\nbfloat16 vs float32:')
print(f'  float32 size: {x_f32.dtype.itemsize * len(x_f32)} bytes')
print(f'  bfloat16 size: {x_bf16.dtype.itemsize * len(x_bf16)} bytes')
print(f'  bfloat16 preserves float32 exponent range (crucial for training stability)')
print(f'  bfloat16 mantissa bits: 7 (vs float16: 10, float32: 23)')

# Demonstrate bfloat16 range vs float16
big = jnp.array(65504.0)  # float16 max
print(f'\nfloat16 overflow test: 65504.0 * 2.0 = {big.astype(jnp.float16) * jnp.array(2.0, jnp.float16)}')
print(f'bfloat16 same op: 65504.0 * 2.0 = {big.astype(jnp.bfloat16) * jnp.array(2.0, jnp.bfloat16)}')
print('(bfloat16 does not overflow where float16 does)')

## 4. TPU Generations: What Changed

| Generation | Year | Matrix Unit | HBM Bandwidth | Notes |
|-----------|------|------------|---------------|-------|
| TPU v2 | 2017 | 128x128 MXU | ~700 GB/s | First cloud TPU |
| TPU v3 | 2018 | 128x128 MXU | ~900 GB/s | 2x compute vs v2 |
| TPU v4 | 2021 | 128x128 MXU | ~1.2 TB/s | SparseCore for embeddings |
| TPU v5e | 2023 | 256x256 MXU | ~1.6 TB/s | LLM inference target |
| TPU v5p | 2023 | 256x256 MXU | ~4.8 TB/s | Large-scale training |

**Key trend**: MXU size doubled from v4 to v5 (128->256), doubling peak matmul throughput. HBM bandwidth increased 4x from v2 to v5p.

The bandwidth increases matter for attention (memory-bound at large sequence lengths) and the MXU size matters for FFN layers (compute-bound).


In [ ]:
# Checking JAX device info (useful on Colab TPU)
for i, device in enumerate(jax.devices()):
    print(f'Device {i}: {device}')
    print(f'  Kind: {device.device_kind}')
    print(f'  Platform: {device.platform}')

# Memory usage (works on TPU/GPU)
def get_array_size_mb(arr):
    return arr.nbytes / 1024**2

x_f32 = jnp.ones((1024, 1024))
x_bf16 = x_f32.astype(jnp.bfloat16)
print(f'\n1024x1024 float32: {get_array_size_mb(x_f32):.1f} MB')
print(f'1024x1024 bfloat16: {get_array_size_mb(x_bf16):.1f} MB')
print(f'bfloat16 saves: {(1 - x_bf16.nbytes/x_f32.nbytes)*100:.0f}% memory')

# Practical takeaway: for a 7B parameter model:
params_7b = 7e9
print(f'\n7B parameter model:')
print(f'  float32: {params_7b * 4 / 1e9:.0f} GB')
print(f'  bfloat16: {params_7b * 2 / 1e9:.0f} GB')
print(f'  int8: {params_7b * 1 / 1e9:.0f} GB')

## Summary

### Key Concepts

| Concept | Why it matters |
|---------|---------------|
| XLA fusion | Elementwise ops merged into one kernel; always use `jit` |
| Static shapes | XLA compiles per shape; pad sequences to avoid recompilation |
| bfloat16 | Native on TPU; same exponent range as float32, avoids overflow |
| Systolic array | Best utilized with large (256+) matrix dimensions |
| HBM bandwidth | Attention is memory-bound; larger batches improve utilization |

**Key design principle**: Write JAX code that maximizes time in the MXU (large matmuls, no branching) and minimizes HBM round-trips (fuse operations, keep activations on-chip). Padding is a feature, not a bug.

**Next**: `08_jax_vs_pytorch_mental_model.ipynb` -- a systematic comparison for people who know PyTorch.
